In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer

from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    classification_report,
    accuracy_score
)

In [3]:
df = pd.read_csv("/content/feedback_multilabel.csv")

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (10500, 3)


,feedback,sentiment,categories
0,The application becomes extremely slow when I ...,negative,performance
1,"For what it's worth, a longer session timeout ...",neutral,login
2,"To be honest, having social login options woul...",neutral,login
3,Biometric login stopped working after the late...,negative,"login,support"
4,Lately everything has been running stably for ...,positive,"bug,feature_request,support"


In [4]:
df["categories"] = df["categories"].apply(
    lambda x: [category.strip() for category in x.split(",")]
)

df.head()

,feedback,sentiment,categories
0,The application becomes extremely slow when I ...,negative,[performance]
1,"For what it's worth, a longer session timeout ...",neutral,[login]
2,"To be honest, having social login options woul...",neutral,[login]
3,Biometric login stopped working after the late...,negative,"[login, support]"
4,Lately everything has been running stably for ...,positive,"[bug, feature_request, support]"


In [5]:
mlb = MultiLabelBinarizer()

y = mlb.fit_transform(df["categories"])

print("Classes:")
print(mlb.classes_)

Classes:
['bug' 'feature_request' 'login' 'payment' 'performance' 'support' 'ui']


In [6]:
y

array([[0, 0, 0, ..., 1, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 1],
       [0, 0, 1, ..., 0, 1, 0],
       [1, 0, 1, ..., 0, 1, 0]])

In [7]:
X = df["feedback"]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 8400
Testing samples: 2100


In [9]:
vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(X_train_tfidf.shape)

(8400, 758)


In [10]:
model = OneVsRestClassifier(
    LogisticRegression(max_iter=1000)
)

model.fit(X_train_tfidf, y_train)

print("Multi-label model trained!")

Multi-label model trained!


In [16]:
y_pred = model.predict(X_test_tfidf)

print(y_pred)

[[1 0 0 ... 0 0 0]
 [0 0 0 ... 1 0 0]
 [1 0 0 ... 1 0 0]
 ...
 [0 0 0 ... 1 1 0]
 [1 0 0 ... 0 1 0]
 [0 0 1 ... 1 1 0]]


In [17]:
print(accuracy_score(y_test, y_pred))

0.9909523809523809


In [12]:
predicted_categories = mlb.inverse_transform(y_pred)

for categories in predicted_categories:
    print(categories)

('bug', 'payment')
('performance',)
('bug', 'performance')
('support', 'ui')
('feature_request',)
('performance', 'ui')
('payment', 'performance', 'ui')
('support', 'ui')
('login', 'ui')
('ui',)
('payment',)
('login',)
('bug', 'login', 'support')
('login', 'support')
('bug', 'performance', 'support')
('feature_request', 'payment')
('feature_request',)
('payment',)
('login', 'payment')
('feature_request', 'login', 'performance')
('payment', 'performance')
('payment', 'ui')
('ui',)
('bug',)
('performance', 'support', 'ui')
('login', 'ui')
('bug', 'login')
('performance', 'ui')
('feature_request', 'payment')
('performance',)
('bug',)
('payment', 'support')
('bug',)
('bug', 'feature_request', 'ui')
('bug',)
('support',)
('feature_request',)
('payment',)
('feature_request',)
('performance',)
('payment',)
('support', 'ui')
('bug', 'ui')
('feature_request', 'performance')
('feature_request', 'login')
('performance', 'ui')
('bug', 'feature_request')
('support', 'ui')
('payment', 'ui')
('featur

In [13]:
new_feedback = [
    "The application is slow and payment keeps failing",
    "I cannot login and the interface button is broken",
    "Payment failed and the application crashed",
    "Please add dark mode"
]

In [14]:
new_feedback_tfidf = vectorizer.transform(new_feedback)

In [15]:
predictions = model.predict(new_feedback_tfidf)

predicted_categories = mlb.inverse_transform(predictions)

for text, categories in zip(new_feedback, predicted_categories):

    print("Feedback:", text)
    print("Categories:", categories)
    print()

Feedback: The application is slow and payment keeps failing
Categories: ('payment',)

Feedback: I cannot login and the interface button is broken
Categories: ('login', 'ui')

Feedback: Payment failed and the application crashed
Categories: ('payment',)

Feedback: Please add dark mode
Categories: ('feature_request',)

